# Agent 2 PostgreSQL Topic-wise Question Inventory

**Purpose:** diagnose retrieval before changing Notebook 05 again.

This notebook is **read-only**. It does not update PostgreSQL, Qdrant, mappings, or retrieval logic.

It prints the raw Agent 2 database question inventory grouped by PMT topic and official AQA mapping, including:

- PMT subtopic code/name
- question-level and mapping-level official references
- Paper 1 / Paper 2
- question ID / number / marks / text
- review status and retrieval flags
- mark-scheme-link availability
- the exact reason a row would be excluded by Notebook 05's PostgreSQL exact-pool filter

The default focus is the current debugging set: `3.6.1`, `3.6.2`, `3.5`, `3.2.11`. Set `FOCUS_REFERENCES = []` to print the whole question bank.

In [1]:
from __future__ import annotations

import os
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

import pandas as pd
from dotenv import load_dotenv
from sqlalchemy import MetaData, Table, create_engine, func, literal, select
from sqlalchemy.engine import Engine


cwd = Path.cwd().resolve()
PROJECT_ROOT = cwd.parent if cwd.name.lower() in {"notebooks", "notebook"} else cwd
OUTPUT_DIR = PROJECT_ROOT / "OUTPUT"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

load_dotenv(PROJECT_ROOT / ".env")

DATABASE_URL = os.getenv("AGENT2_DATABASE_URL", "").strip()
if not DATABASE_URL:
    raise RuntimeError("AGENT2_DATABASE_URL is missing from Agent2/.env")

# Current debugging references. Use [] to include the entire Agent 2 bank.
FOCUS_REFERENCES = ["3.6.1", "3.6.2", "3.5", "3.2.11"]

engine: Engine = create_engine(
    DATABASE_URL,
    pool_pre_ping=True,
    future=True,
)
metadata = MetaData()

topics = Table(
    "assessment_topical_topics",
    metadata,
    autoload_with=engine,
)
questions = Table(
    "assessment_topical_questions",
    metadata,
    autoload_with=engine,
)
question_ms_links = Table(
    "assessment_topical_question_mark_scheme_links",
    metadata,
    autoload_with=engine,
)
official_mappings = Table(
    "assessment_topic_official_mappings",
    metadata,
    autoload_with=engine,
)

def optional_column(table: Table, names: list[str]):
    for name in names:
        if name in table.c:
            return table.c[name]
    return None


def selected_or_null(column, label: str):
    return column.label(label) if column is not None else literal(None).label(label)


# Question optional columns used by Notebook 05.
q_question_uid = optional_column(questions, ["question_uid", "uid"])
q_context = optional_column(questions, ["context_text"])
q_retrieval_enabled = optional_column(questions, ["retrieval_enabled"])
q_is_active = optional_column(questions, ["is_active"])
q_is_legacy = optional_column(questions, ["is_legacy"])
q_record_type = optional_column(questions, ["record_type"])
q_page_start = optional_column(questions, ["page_start"])
q_page_end = optional_column(questions, ["page_end"])

# Topic columns.
t_pmt_code = optional_column(topics, ["pmt_subtopic_code"])
t_pmt_name = optional_column(topics, ["pmt_subtopic_name", "topic_name", "name"])
t_paper = optional_column(topics, ["paper_code"])
t_language = optional_column(topics, ["programming_language"])

# Mapping columns.
m_map_status = optional_column(official_mappings, ["mapping_status"])
m_human_approved = optional_column(official_mappings, ["human_approved"])
m_pmt_code = optional_column(official_mappings, ["pmt_subtopic_code"])
m_pmt_name = optional_column(official_mappings, ["pmt_subtopic_name"])
m_official_reference = optional_column(official_mappings, ["official_reference"])
m_official_concept = optional_column(official_mappings, ["official_concept_name"])
m_section_reference = optional_column(official_mappings, ["official_section_reference"])
m_section_name = optional_column(official_mappings, ["official_section_name"])

with engine.connect() as connection:
    connection.exec_driver_sql("SELECT 1")

print("PostgreSQL connection successful.")
print("Question columns:", list(questions.c.keys()))
print("Topic columns:", list(topics.c.keys()))
print("Mapping columns:", list(official_mappings.c.keys()))


PostgreSQL connection successful.
Question columns: ['id', 'question_uid', 'pair_key', 'topic_id', 'question_document_id', 'sequence_index', 'question_number', 'normalized_question_number', 'occurrence_index', 'main_question_number', 'part_number', 'parent_question_number', 'record_type', 'question_text', 'context_text', 'search_text', 'raw_extracted_text', 'marks', 'page_start', 'page_end', 'has_visual', 'visual_page_numbers', 'has_code', 'specification_scope', 'is_legacy', 'parse_warnings', 'review_status', 'retrieval_enabled', 'embedding_status', 'content_hash', 'parse_version', 'is_active', 'created_at', 'updated_at', 'official_specification_code', 'official_specification_version', 'official_section_reference', 'official_section_name', 'official_reference', 'official_concept_name', 'official_reference_level', 'official_mapping_version', 'official_mapped_at']
Topic columns: ['id', 'topic_key', 'source_provider', 'source_topic_page_url', 'exam_board', 'qualification', 'subject', 'tar

In [2]:
# ---------------------------------------------------------
# Read the database without applying Notebook 05 retrieval filters.
# ---------------------------------------------------------

ms_link_counts = (
    select(
        question_ms_links.c.question_id.label("question_id"),
        func.count().label("mark_scheme_link_count"),
    )
    .group_by(question_ms_links.c.question_id)
    .subquery()
)

question_query = (
    select(
        questions.c.id.label("question_id"),
        selected_or_null(q_question_uid, "question_uid"),
        questions.c.topic_id.label("topic_id"),
        selected_or_null(t_pmt_code, "pmt_subtopic_code"),
        selected_or_null(t_pmt_name, "pmt_subtopic_name"),
        selected_or_null(t_paper, "paper_code"),
        selected_or_null(t_language, "programming_language"),
        questions.c.question_number.label("question_number"),
        questions.c.marks.label("marks"),
        questions.c.question_text.label("question_text"),
        selected_or_null(q_context, "context_text"),
        questions.c.official_reference.label("question_official_reference"),
        questions.c.official_concept_name.label("question_official_concept_name"),
        questions.c.official_section_reference.label("question_official_section_reference"),
        questions.c.review_status.label("review_status"),
        questions.c.has_code.label("has_code"),
        questions.c.has_visual.label("has_visual"),
        selected_or_null(q_retrieval_enabled, "retrieval_enabled"),
        selected_or_null(q_is_active, "is_active"),
        selected_or_null(q_is_legacy, "is_legacy"),
        selected_or_null(q_record_type, "record_type"),
        selected_or_null(q_page_start, "page_start"),
        selected_or_null(q_page_end, "page_end"),
        func.coalesce(ms_link_counts.c.mark_scheme_link_count, 0).label(
            "mark_scheme_link_count"
        ),
    )
    .select_from(
        questions
        .join(topics, questions.c.topic_id == topics.c.id)
        .outerjoin(ms_link_counts, ms_link_counts.c.question_id == questions.c.id)
    )
    .order_by(
        t_pmt_code if t_pmt_code is not None else topics.c.id,
        t_paper if t_paper is not None else topics.c.id,
        questions.c.question_number,
        questions.c.id,
    )
)

mapping_query = select(
    official_mappings.c.topic_id.label("topic_id"),
    selected_or_null(m_pmt_code, "mapping_pmt_subtopic_code"),
    selected_or_null(m_pmt_name, "mapping_pmt_subtopic_name"),
    selected_or_null(m_official_reference, "mapping_official_reference"),
    selected_or_null(m_official_concept, "mapping_official_concept_name"),
    selected_or_null(m_section_reference, "mapping_official_section_reference"),
    selected_or_null(m_section_name, "mapping_official_section_name"),
    selected_or_null(m_map_status, "mapping_status"),
    selected_or_null(m_human_approved, "mapping_human_approved"),
)

with engine.connect() as connection:
    questions_df = pd.read_sql(question_query, connection)
    mappings_df = pd.read_sql(mapping_query, connection)

# One mapping row per topic is expected. If history/duplicates exist, retain all
# source questions but choose the most recently returned unique topic mapping
# for this diagnostic view.
mappings_df = mappings_df.drop_duplicates(subset=["topic_id"], keep="last")

inventory_df = questions_df.merge(
    mappings_df,
    on="topic_id",
    how="left",
    validate="many_to_one",
)


def notebook05_exclusion_reasons(row: pd.Series) -> list[str]:
    """Reproduce only the DB eligibility gates used by postgres_exact_rows()."""
    reasons: list[str] = []

    if "review_status" in inventory_df.columns:
        review_status = str(row.get("review_status") or "").strip()
        if review_status not in {"human_approved", "human_corrected"}:
            reasons.append(f"review_status={review_status or '<blank>'}")

    if q_retrieval_enabled is not None and row.get("retrieval_enabled") is not True:
        reasons.append(f"retrieval_enabled={row.get('retrieval_enabled')}")

    if q_is_active is not None and row.get("is_active") is not True:
        reasons.append(f"is_active={row.get('is_active')}")

    if q_is_legacy is not None and row.get("is_legacy") is not False:
        reasons.append(f"is_legacy={row.get('is_legacy')}")

    if q_record_type is not None:
        record_type = str(row.get("record_type") or "").strip()
        if record_type != "scored_item":
            reasons.append(f"record_type={record_type or '<blank>'}")

    if int(row.get("mark_scheme_link_count") or 0) <= 0:
        reasons.append("no_mark_scheme_link")

    return reasons


inventory_df["db_retrieval_exclusion_reasons"] = inventory_df.apply(
    notebook05_exclusion_reasons,
    axis=1,
)
inventory_df["db_retrieval_eligible"] = inventory_df[
    "db_retrieval_exclusion_reasons"
].map(lambda values: len(values) == 0)

# Explicitly surface disagreement between the raw question metadata and the
# topic->official mapping table.
inventory_df["reference_mismatch"] = (
    inventory_df["question_official_reference"].fillna("").astype(str).str.strip()
    != inventory_df["mapping_official_reference"].fillna("").astype(str).str.strip()
)

if FOCUS_REFERENCES:
    reference_columns = [
        inventory_df["question_official_reference"].fillna("").astype(str),
        inventory_df["mapping_official_reference"].fillna("").astype(str),
    ]
    focus_mask = reference_columns[0].isin(FOCUS_REFERENCES) | reference_columns[1].isin(
        FOCUS_REFERENCES
    )
    report_df = inventory_df.loc[focus_mask].copy()
else:
    report_df = inventory_df.copy()

report_df = report_df.sort_values(
    [
        "mapping_official_reference",
        "pmt_subtopic_code",
        "paper_code",
        "question_number",
        "question_id",
    ],
    na_position="last",
).reset_index(drop=True)

print(f"All DB question rows: {len(inventory_df):,}")
print(f"Rows included in diagnostic report: {len(report_df):,}")
print(f"DB-retrieval eligible rows in report: {int(report_df['db_retrieval_eligible'].sum()):,}")
print(f"Reference mismatches in report: {int(report_df['reference_mismatch'].sum()):,}")

summary_df = (
    report_df.groupby(
        [
            "mapping_official_reference",
            "mapping_official_concept_name",
            "pmt_subtopic_code",
            "pmt_subtopic_name",
            "paper_code",
        ],
        dropna=False,
    )
    .agg(
        questions=("question_id", "count"),
        db_eligible=("db_retrieval_eligible", "sum"),
        reference_mismatches=("reference_mismatch", "sum"),
    )
    .reset_index()
)

display(summary_df)


All DB question rows: 942
Rows included in diagnostic report: 131
DB-retrieval eligible rows in report: 118
Reference mismatches in report: 0


,mapping_official_reference,mapping_official_concept_name,pmt_subtopic_code,pmt_subtopic_name,paper_code,questions,db_eligible,reference_mismatches
0,3.2.11,Robust and secure programming,2.11,Robust and Secure Programming,1,66,56,0
1,3.5,Fundamentals of computer networks,5,Fundamentals of Computer Networks,2,47,44,0
2,3.6.1,Fundamentals of cyber security,6.1,Fundamentals of Cyber Security,2,1,1,0
3,3.6.2,Cyber security threats,6.2,Cyber Security Threats,2,17,17,0


In [3]:
# ---------------------------------------------------------
# Quick console checks for the two currently suspicious concepts.
# ---------------------------------------------------------

search_terms = [
    "cyber security",
    "network protocol",
    "smtp",
    "imap",
    "encryption",
]

for term in search_terms:
    matches = report_df[
        report_df["question_text"]
        .fillna("")
        .astype(str)
        .str.contains(term, case=False, regex=False)
    ]

    print("\n" + "=" * 100)
    print(f"SEARCH TERM: {term!r} | matches: {len(matches)}")
    print("=" * 100)

    if matches.empty:
        print("No DB row found in the focused report.")
        continue

    display(
        matches[
            [
                "question_id",
                "pmt_subtopic_code",
                "pmt_subtopic_name",
                "paper_code",
                "question_number",
                "marks",
                "question_official_reference",
                "question_official_concept_name",
                "mapping_official_reference",
                "mapping_official_concept_name",
                "review_status",
                "mark_scheme_link_count",
                "db_retrieval_eligible",
                "db_retrieval_exclusion_reasons",
                "question_text",
            ]
        ]
    )



SEARCH TERM: 'cyber security' | matches: 5


,question_id,pmt_subtopic_code,pmt_subtopic_name,paper_code,question_number,marks,question_official_reference,question_official_concept_name,mapping_official_reference,mapping_official_concept_name,review_status,mark_scheme_link_count,db_retrieval_eligible,db_retrieval_exclusion_reasons,question_text
113,2887db8f-b580-46fc-93c7-1ea7d6e74e9f,6.1,Fundamentals of Cyber Security,2,03.1,2.0,3.6.1,Fundamentals of cyber security,3.6.1,Fundamentals of cyber security,human_approved,1,True,[],Define the term cyber security.
117,5dc6657c-1773-4b4c-8dc3-4c6ac2c57ed4,6.2,Cyber Security Threats,2,04.1,2.0,3.6.2,Cyber security threats,3.6.2,Cyber security threats,human_approved,1,True,[],Define the term ‘cyber security’.
119,d5fb3203-e641-4af4-9c98-4e2b18fb076e,6.2,Cyber Security Threats,2,04.3,8.0,3.6.2,Cyber security threats,3.6.2,Cyber security threats,human_approved,1,True,[],Explain how each of the following cyber securi...
125,c73b1d99-44d5-4a46-bcb3-aef4f686099d,6.2,Cyber Security Threats,2,07.1,2.0,3.6.2,Cyber security threats,3.6.2,Cyber security threats,human_approved,1,True,[],Define the term cyber security.
127,558a731e-f094-497e-b449-28741ce8b76c,6.2,Cyber Security Threats,2,07.3,9.0,3.6.2,Cyber security threats,3.6.2,Cyber security threats,human_approved,1,True,[],The network manager of a new computer games co...



SEARCH TERM: 'network protocol' | matches: 4


,question_id,pmt_subtopic_code,pmt_subtopic_name,paper_code,question_number,marks,question_official_reference,question_official_concept_name,mapping_official_reference,mapping_official_concept_name,review_status,mark_scheme_link_count,db_retrieval_eligible,db_retrieval_exclusion_reasons,question_text
76,c9f75cb8-4b6f-475d-a20b-cc3769dba0d0,5,Fundamentals of Computer Networks,2,02.6,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Define the term network protocol.
84,5a5d9d8c-4eb6-45e2-bd85-d02a159be330,5,Fundamentals of Computer Networks,2,05.3,4.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],State which layer of the TCP/IP model each of ...
94,95ff15b4-9bbd-4be8-9296-393d38ff694b,5,Fundamentals of Computer Networks,2,07.4,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],HTTP is an example of a network protocol.\nDef...
101,56277dbf-014a-4cdc-a0db-0ad4d1c276d8,5,Fundamentals of Computer Networks,2,08.5,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],When two computers on a network communicate wi...



SEARCH TERM: 'smtp' | matches: 6


,question_id,pmt_subtopic_code,pmt_subtopic_name,paper_code,question_number,marks,question_official_reference,question_official_concept_name,mapping_official_reference,mapping_official_concept_name,review_status,mark_scheme_link_count,db_retrieval_eligible,db_retrieval_exclusion_reasons,question_text
77,6dd3a61b-7498-4e8c-a4c0-20f4c69b1ca9,5,Fundamentals of Computer Networks,2,02.7,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Which two of the following are email protocols...
89,a4fb7dfc-23df-48bf-bf1a-bdf399899911,5,Fundamentals of Computer Networks,2,06.4,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Shade one lozenge to indicate the application ...
102,a53919ea-3375-42ea-9476-ba03f8ea6cc7,5,Fundamentals of Computer Networks,2,08.6,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Used to retrieve email stored on a server\n\nA...
103,e5b04ac7-f9f9-4f92-9465-89d65804e97e,5,Fundamentals of Computer Networks,2,08.7,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Used to make a payment securely when purchasin...
104,522234bc-45af-49dc-885c-ee761152d944,5,Fundamentals of Computer Networks,2,08.8,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Used to send an email from a client machine to...
111,6634e32d-5ea8-467b-9ed2-49e6d973423a,5,Fundamentals of Computer Networks,2,11,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],SMTP and IMAP are email protocols.\nDescribe h...



SEARCH TERM: 'imap' | matches: 6


,question_id,pmt_subtopic_code,pmt_subtopic_name,paper_code,question_number,marks,question_official_reference,question_official_concept_name,mapping_official_reference,mapping_official_concept_name,review_status,mark_scheme_link_count,db_retrieval_eligible,db_retrieval_exclusion_reasons,question_text
77,6dd3a61b-7498-4e8c-a4c0-20f4c69b1ca9,5,Fundamentals of Computer Networks,2,02.7,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Which two of the following are email protocols...
84,5a5d9d8c-4eb6-45e2-bd85-d02a159be330,5,Fundamentals of Computer Networks,2,05.3,4.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],State which layer of the TCP/IP model each of ...
102,a53919ea-3375-42ea-9476-ba03f8ea6cc7,5,Fundamentals of Computer Networks,2,08.6,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Used to retrieve email stored on a server\n\nA...
103,e5b04ac7-f9f9-4f92-9465-89d65804e97e,5,Fundamentals of Computer Networks,2,08.7,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Used to make a payment securely when purchasin...
104,522234bc-45af-49dc-885c-ee761152d944,5,Fundamentals of Computer Networks,2,08.8,1.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Used to send an email from a client machine to...
111,6634e32d-5ea8-467b-9ed2-49e6d973423a,5,Fundamentals of Computer Networks,2,11,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],SMTP and IMAP are email protocols.\nDescribe h...



SEARCH TERM: 'encryption' | matches: 1


,question_id,pmt_subtopic_code,pmt_subtopic_name,paper_code,question_number,marks,question_official_reference,question_official_concept_name,mapping_official_reference,mapping_official_concept_name,review_status,mark_scheme_link_count,db_retrieval_eligible,db_retrieval_exclusion_reasons,question_text
112,66190430-2b6c-4b4d-af13-8286e9619b21,5,Fundamentals of Computer Networks,2,12,2.0,3.5,Fundamentals of computer networks,3.5,Fundamentals of computer networks,human_approved,1,True,[],Describe how encryption can make the transmiss...


In [4]:
# ---------------------------------------------------------
# Export CSV + topic-wise diagnostic PDF.
# ---------------------------------------------------------

from xml.sax.saxutils import escape

from reportlab.lib import colors
from reportlab.lib.enums import TA_CENTER
from reportlab.lib.pagesizes import A4
from reportlab.lib.styles import ParagraphStyle, getSampleStyleSheet
from reportlab.lib.units import mm
from reportlab.platypus import (
    PageBreak,
    Paragraph,
    SimpleDocTemplate,
    Spacer,
    Table,
    TableStyle,
)

stamp = datetime.now(timezone.utc).strftime("%Y%m%d_%H%M%S")

csv_path = OUTPUT_DIR / f"agent2_db_topic_question_inventory_{stamp}.csv"
pdf_path = OUTPUT_DIR / f"agent2_db_topic_question_inventory_{stamp}.pdf"

csv_export_df = report_df.copy()
csv_export_df["db_retrieval_exclusion_reasons"] = csv_export_df[
    "db_retrieval_exclusion_reasons"
].map(lambda values: "; ".join(values))
csv_export_df.to_csv(csv_path, index=False, encoding="utf-8-sig")

styles = getSampleStyleSheet()
styles.add(
    ParagraphStyle(
        name="DiagTitle",
        parent=styles["Title"],
        alignment=TA_CENTER,
        fontName="Helvetica-Bold",
        fontSize=17,
        leading=21,
        spaceAfter=12,
        textColor=colors.black,
    )
)
styles.add(
    ParagraphStyle(
        name="DiagTopic",
        parent=styles["Heading2"],
        fontName="Helvetica-Bold",
        fontSize=12,
        leading=15,
        spaceBefore=8,
        spaceAfter=5,
        textColor=colors.black,
    )
)
styles.add(
    ParagraphStyle(
        name="DiagQuestion",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=9,
        leading=12,
        spaceAfter=4,
        textColor=colors.black,
    )
)
styles.add(
    ParagraphStyle(
        name="DiagSmall",
        parent=styles["BodyText"],
        fontName="Helvetica",
        fontSize=7.4,
        leading=9.5,
        textColor=colors.black,
    )
)


def p(value: Any, style="DiagSmall") -> Paragraph:
    text_value = "" if pd.isna(value) else str(value)
    return Paragraph(escape(text_value).replace("\n", "<br/>"), styles[style])


def yn(value: Any) -> str:
    return "Yes" if bool(value) else "No"


def page_number(canvas, doc):
    canvas.saveState()
    canvas.setFont("Helvetica", 8)
    canvas.drawRightString(A4[0] - 15 * mm, 10 * mm, f"Page {doc.page}")
    canvas.restoreState()


doc = SimpleDocTemplate(
    str(pdf_path),
    pagesize=A4,
    rightMargin=14 * mm,
    leftMargin=14 * mm,
    topMargin=14 * mm,
    bottomMargin=16 * mm,
    title="Agent 2 PostgreSQL Topic-wise Question Inventory",
)

story = []
story.append(p("Agent 2 PostgreSQL Topic-wise Question Inventory", "DiagTitle"))
story.append(
    p(
        "Read-only diagnostic export. No retrieval, mapping, PostgreSQL, or Qdrant records were modified.",
        "DiagQuestion",
    )
)
story.append(
    p(
        f"Generated UTC: {datetime.now(timezone.utc).isoformat()} | "
        f"Focus references: {', '.join(FOCUS_REFERENCES) if FOCUS_REFERENCES else 'ALL'} | "
        f"Rows: {len(report_df)} | DB-eligible: {int(report_df['db_retrieval_eligible'].sum())}",
        "DiagSmall",
    )
)
story.append(Spacer(1, 5 * mm))

summary_table_data = [
    [
        p("Official ref"),
        p("Official concept"),
        p("PMT topic"),
        p("Paper"),
        p("Questions"),
        p("Eligible"),
        p("Ref mismatch"),
    ]
]

for _, row in summary_df.iterrows():
    summary_table_data.append(
        [
            p(row.get("mapping_official_reference")),
            p(row.get("mapping_official_concept_name")),
            p(
                f"{row.get('pmt_subtopic_code') or ''} "
                f"{row.get('pmt_subtopic_name') or ''}".strip()
            ),
            p(row.get("paper_code")),
            p(row.get("questions")),
            p(row.get("db_eligible")),
            p(row.get("reference_mismatches")),
        ]
    )

summary_table = Table(
    summary_table_data,
    colWidths=[17 * mm, 43 * mm, 50 * mm, 14 * mm, 16 * mm, 15 * mm, 19 * mm],
    repeatRows=1,
)
summary_table.setStyle(
    TableStyle(
        [
            ("GRID", (0, 0), (-1, -1), 0.35, colors.black),
            ("FONTNAME", (0, 0), (-1, 0), "Helvetica-Bold"),
            ("VALIGN", (0, 0), (-1, -1), "TOP"),
            ("LEFTPADDING", (0, 0), (-1, -1), 3),
            ("RIGHTPADDING", (0, 0), (-1, -1), 3),
            ("TOPPADDING", (0, 0), (-1, -1), 3),
            ("BOTTOMPADDING", (0, 0), (-1, -1), 3),
        ]
    )
)
story.append(summary_table)
story.append(PageBreak())

# Group first by actual PMT topic; this shows what the bank physically contains.
group_cols = [
    "topic_id",
    "pmt_subtopic_code",
    "pmt_subtopic_name",
    "paper_code",
    "programming_language",
    "mapping_official_reference",
    "mapping_official_concept_name",
]

for group_index, (group_key, group_df) in enumerate(
    report_df.groupby(group_cols, dropna=False, sort=False),
    start=1,
):
    (
        topic_id,
        pmt_code,
        pmt_name,
        paper_code,
        programming_language,
        mapping_ref,
        mapping_concept,
    ) = group_key

    story.append(
        p(
            f"PMT {pmt_code or ''} - {pmt_name or 'Unnamed topic'}",
            "DiagTopic",
        )
    )
    story.append(
        p(
            f"topic_id={topic_id} | paper={paper_code or 'N/A'} | "
            f"language={programming_language or 'N/A'} | "
            f"mapping={mapping_ref or 'N/A'} - {mapping_concept or 'N/A'} | "
            f"questions={len(group_df)} | "
            f"DB-eligible={int(group_df['db_retrieval_eligible'].sum())}",
            "DiagSmall",
        )
    )
    story.append(Spacer(1, 2 * mm))

    for _, row in group_df.iterrows():
        exclusion_reasons = row.get("db_retrieval_exclusion_reasons") or []
        eligibility_text = (
            "ELIGIBLE for Notebook 05 PostgreSQL exact pool"
            if bool(row.get("db_retrieval_eligible"))
            else "EXCLUDED: " + "; ".join(str(v) for v in exclusion_reasons)
        )

        meta_data = [
            [p("Q ID"), p(row.get("question_id")), p("Q No"), p(row.get("question_number")), p("Marks"), p(row.get("marks"))],
            [p("Question ref"), p(row.get("question_official_reference")), p("Question concept"), p(row.get("question_official_concept_name")), p("MS links"), p(row.get("mark_scheme_link_count"))],
            [p("Review"), p(row.get("review_status")), p("Eligible?"), p(yn(row.get("db_retrieval_eligible"))), p("Ref mismatch?"), p(yn(row.get("reference_mismatch")))],
            [p("Flags"), p(f"retrieval_enabled={row.get('retrieval_enabled')} | is_active={row.get('is_active')} | is_legacy={row.get('is_legacy')} | record_type={row.get('record_type')}") , p("Pages"), p(f"{row.get('page_start')} - {row.get('page_end')}"), p("Visual/code"), p(f"visual={row.get('has_visual')} | code={row.get('has_code')}")],
        ]

        meta_table = Table(
            meta_data,
            colWidths=[20 * mm, 24 * mm, 18 * mm, 47 * mm, 18 * mm, 47 * mm],
        )
        meta_table.setStyle(
            TableStyle(
                [
                    ("GRID", (0, 0), (-1, -1), 0.25, colors.black),
                    ("FONTNAME", (0, 0), (-1, -1), "Helvetica"),
                    ("VALIGN", (0, 0), (-1, -1), "TOP"),
                    ("LEFTPADDING", (0, 0), (-1, -1), 2),
                    ("RIGHTPADDING", (0, 0), (-1, -1), 2),
                    ("TOPPADDING", (0, 0), (-1, -1), 2),
                    ("BOTTOMPADDING", (0, 0), (-1, -1), 2),
                ]
            )
        )
        story.append(meta_table)
        story.append(Spacer(1, 1.5 * mm))
        story.append(p(f"<b>DB retrieval status:</b> {eligibility_text}", "DiagSmall"))
        story.append(p(f"<b>Question:</b> {row.get('question_text') or ''}", "DiagQuestion"))

        context = str(row.get("context_text") or "").strip()
        if context:
            story.append(p(f"Context: {context}", "DiagSmall"))

        story.append(Spacer(1, 3.5 * mm))

    if group_index < report_df[group_cols].drop_duplicates().shape[0]:
        story.append(PageBreak())

doc.build(story, onFirstPage=page_number, onLaterPages=page_number)

print("Diagnostic CSV:", csv_path)
print("Diagnostic PDF:", pdf_path)
print("PDF bytes:", pdf_path.stat().st_size)


Diagnostic CSV: C:\Users\hp\EDTECH\Agent2\OUTPUT\agent2_db_topic_question_inventory_20260807_123852.csv
Diagnostic PDF: C:\Users\hp\EDTECH\Agent2\OUTPUT\agent2_db_topic_question_inventory_20260807_123852.pdf
PDF bytes: 131400
